In [ ]:
import pandas as pd

import numpy as np

import os

from sklearn.model_selection import StratifiedKFold, train_test_split

In [ ]:
MAX_SEQ_LEN = 8427

MAX_TIME = 365

K = 5

TEST_SIZE = 0.2

FACTOR = 1/30

In [ ]:
def isEligible(x:pd.Series) -> int:
    if 'NFALG' in x or 'NFGEH' in x or 'NFVMI+' in x or 'NFVAL+' in x:
        return 1
    else:
        return 0

# Raggruppare per paziente, ordinare per data e filtrare in base alla condizione
def filter_positive_samples(group: pd.Series):
    # Se uno dei campioni è positivo, mantieni tutti i campioni precedenti
    if 1 in group['label'].values:
        # Troviamo l'indice del primo campione positivo
        first_positive_index = group[group['label'] == 1].index[0]
        # Mantieni tutti i campioni fino a questo indice incluso
        return group.loc[:first_positive_index]
    
    else:
        # Se non ci sono campioni positivi, mantieni tutti
        return group


def avg_mean_emb(group: pd.DataFrame):
    # Seleziona i dati numerici (gli embeddings delle medicazioni)
    numeric_data = group.select_dtypes(include=np.float32)
    
    # Verifica se ci sono NaN nei dati numerici (embeddings)
    if numeric_data.isna().any().any():  # Se c'è almeno un NaN in qualche colonna degli embeddings
        # Restituisci NaN per tutte le colonne degli embeddings
        output = group[['studyId_0831', 'recent_date', 'age', 'gender', 'consult', 'letter']].iloc[0]
        # Crea una Serie di NaN con lo stesso numero di colonne degli embeddings
        output = pd.concat([output, pd.Series([np.nan] * len(numeric_data.columns), index=numeric_data.columns)])
        return output.to_frame().T  # Trasforma in DataFrame e ritorna una riga
    
    # Se non ci sono NaN, calcola la media pesata come prima
    delta = group['delta']
    
    weights = np.exp(-delta * FACTOR)
    weights = weights / weights.sum()  # Normalizza i pesi
    
    # Calcola la media pesata
    weighted_avg = (numeric_data.T * weights).T.sum()
    
    # Seleziona le colonne per cui hai fatto il groupby
    output = group[['studyId_0831', 'recent_date', 'age', 'gender', 'consult', 'letter']].iloc[0]
    
    # Aggiungi la media pesata degli embeddings all'output
    output = pd.concat([output, weighted_avg])
    
    # Restituisci una riga di DataFrame con l'output
    return output.to_frame().T

In [ ]:
def generate_merge_dataset(
        df          : pd.DataFrame, 
        random_seed : int = 42,
        test_size   : float = 0.2,
        n_splits    : int = 5,
        root        : str = ''
    ):
    
    '''
    This is the new version for whole text dataset.
    Parameters
    ---
    - df: dataframe with two columns: text (str) and labels make sure to reset index
    - test_split: test set percentage
    - n_splits: number of folds for the k-fold cross-validation
    - root: path/to/root/containing the k-fold cross-validation
    '''
    
    assert n_splits >= 2, "n_splits needs to be greater than 1"
    
    os.makedirs(root, exist_ok=True)
    
    os.chdir(root)
    
    test_path = os.path.join(root, 'Test')
    os.makedirs(test_path, exist_ok=True)
    
    df_trainval, df_test = train_test_split(
        df,
        test_size = test_size,
        stratify = df.iloc[:, -1],
        random_state = random_seed
    )
    
    
    for _, sample in df_test.iterrows():
        sample = sample.to_frame().T
        
        #print(sample.columns)
        
        sample_name = f"{sample['studyId_0831'].iloc[0]}.parquet"
        
        sample.to_parquet(os.path.join(test_path, sample_name))
    
    
    skf = StratifiedKFold(
        n_splits = n_splits,
        shuffle = True,
        random_state = random_seed
    )
    
    print(f'Starting {n_splits}-fold splitting...')
    
    for split, (train_idx, val_idx) in enumerate(skf.split(df_trainval, df_trainval.iloc[:, -1])):
        # Creazione della cartella per lo split i-esimo
        split_path = os.path.join(root, f'split_{split}')
        os.makedirs(split_path, exist_ok=True)
        
        # Creazione della cartella training
        train_path = os.path.join(split_path, 'train')
        os.makedirs(train_path, exist_ok=True)
        
        # Salvo i campioni di training
        for idx in train_idx:
            train_sample = df_trainval.iloc[[idx]]
            
            sample_name = f'{train_sample["studyId_0831"].values[0]}.parquet'
            
            #print(sample_name)
            
            train_sample.to_parquet(os.path.join(train_path, sample_name))
        
        # Creazione della cartella di validation
        val_path = os.path.join(split_path, 'val')
        os.makedirs(val_path, exist_ok=True)
        
        # Salvo i campioni di validation
        for idx in val_idx:
            val_sample = df_trainval.iloc[[idx]]
            
            sample_name = f'{val_sample["studyId_0831"].values[0]}.parquet'
            
            #print(sample_name)
            
            val_sample.to_parquet(os.path.join(val_path, sample_name))
        
    print(f'{n_splits}-fold cross-validation completed successfully')

## 1. Pre-processing of the patients agenda codes

In [ ]:
df_afsprak = pd.read_sas(
    r'T:\lab_research\RES-Folder-UPOD\ODIN-UC4\E_ResearchData\2_ResearchData\20240805\afspraken_20240805.sas7bdat',
    encoding='latin1'
)

df_afsprak['studyId_0831'] = df_afsprak['studyId_0831'].astype('uint64')

df_afsprak = df_afsprak.filter(items=['studyId_0831', 'index_AfspCode', 'period_start'])

df_afsprak = df_afsprak.sort_values(by=['studyId_0831', 'period_start'])

df_afsprak = df_afsprak.rename(columns={'period_start': 'date'})

# Rimuovo l'orario dalla data
df_afsprak['date'] = df_afsprak['date'].dt.normalize()

df_afsprak = df_afsprak.reset_index(drop=True)

df_afsprak

### 1.1 Viene applicata la funzione `is_Eligible` per assegnare la label ad ogni paziente

In [ ]:
df_afsprak = df_afsprak.drop_duplicates()

# Applico la funzione "isEligible" alla colonna degli agendacodes
df_afsprak['label'] = df_afsprak['index_AfspCode'].apply(isEligible)

df_afsprak

### 1.2 Dataframe filtering
For each patient, the sample corresponding to the first insertion date into the CVRM group is retained, along with any previous samples if available. If the patient does not belong to the CVRM group, the most recent appointment date is used. To consider the most relevant patient history, all samples with a time difference greater than `MAX_TIME` from the most recent date are discarded.

In [ ]:
# Applicare la funzione di filtraggio al gruppo di pazienti
df_afsprak_filt = df_afsprak.groupby('studyId_0831')[['studyId_0831', 'date', 'label']].apply(filter_positive_samples).reset_index(drop=True)

df_afsprak_filt

In [ ]:
# Calcolo la data più recente (ultima data) per ogni paziente
df_afsprak_filt['last_date'] = df_afsprak_filt.groupby('studyId_0831')['date'].transform('max')

# Calcolo la differenza in giorni tra l'ultima data e tutte le altre date per ogni paziente
df_afsprak_filt['delta'] = (df_afsprak_filt['last_date'] - df_afsprak_filt['date']).dt.days

df_afsprak_filt

In [ ]:
# Filtro i campioni che hanno una differenza maggiore di MAX_TIME
df_afsprak_recents = df_afsprak_filt[df_afsprak_filt['delta'] <= MAX_TIME].reset_index(drop=True)

df_afsprak_recents = df_afsprak_recents.drop(columns=['last_date', 'delta'])

# Rimuovo i campioni con entrate precedenti alla prima data utile
df_afsprak_recents = df_afsprak_recents.drop_duplicates(
    subset = 'studyId_0831',
    keep = 'last'
)

df_afsprak_recents = df_afsprak_recents.rename(columns={'date': 'recent_date'})

df_afsprak_recents

In [ ]:
labels = df_afsprak_recents[['studyId_0831', 'label']]

labels

## 2. Aggiunta dell'età e del sesso dei pazienti

1 = Female 

0 = Male

In [ ]:
df_age_gender = pd.read_parquet(r'T:\lab_research\RES-Folder-UPOD\ODIN-UC4\G_Output\2_Data\parquet\df_age_gender.parquet')

df_age_gender

In [ ]:
num_missing_in_age_gender = len(set(df_afsprak_recents['studyId_0831']) - set(df_age_gender['studyId_0831']))

missing_in_age_gender = list(set(df_afsprak_recents['studyId_0831']) - set(df_age_gender['studyId_0831']))

print(f"Patients with missing age and gender ({num_missing_in_age_gender}): \n{missing_in_age_gender}")

In [ ]:
df_1_2 = pd.merge(
    left = df_afsprak_recents,
    right = df_age_gender,
    on = 'studyId_0831',
    how = 'left'
)

df_1_2.drop(columns='label', inplace=True)

df_1_2

## 3. Aggiunta dei consulti dei pazienti

In [ ]:
df_consults = pd.read_parquet(r"T:\lab_research\RES-Folder-UPOD\ODIN-UC4\E_ResearchData\2_ResearchData\20240826\consult_texts.parquet")

df_consults['studyId_0831'] = df_consults['studyId_0831'].astype('uint64')

# Rimuovo l'orario dalla data
df_consults['date'] = df_consults['date'].dt.normalize()

df_consults.drop_duplicates(inplace=True)

df_consults.rename(columns={'text': 'consult'}, inplace=True)

df_consults = df_consults.sort_values(by=['studyId_0831', 'date']).reset_index(drop=True)

df_consults

Cell for agendacodes and consults date verification

In [ ]:
#df_consults = df_consults.assign(has_cvrm=df_consults.text.str.contains(r'CVRM|cardiovasculair risicomanagement', regex=True, case=True))

#df_consults

In [ ]:
num_missing_in_consults = len(set(df_1_2['studyId_0831']) - set(df_consults['studyId_0831']))

missing_in_consults = list(set(df_1_2['studyId_0831']) - set(df_consults['studyId_0831']))

print(f"Patients with missing consult ({num_missing_in_consults}): \n{missing_in_consults}")

In [ ]:
df_1_2_3 = pd.merge(
    left = df_1_2,
    right = df_consults,
    on = 'studyId_0831',
    how = 'left'
)

df_1_2_3

In [ ]:
# Calcolo la differenza in giorni tra l'ultima data e tutte le altre date per ogni paziente
df_1_2_3['delta'] = (df_1_2_3['recent_date'] - df_1_2_3['date']).dt.days

df_1_2_3

In [ ]:
df_1_2_3.loc[~df_1_2_3['delta'].between(0, MAX_TIME, inclusive='both'), 'consult'] = None

df_1_2_3

In [ ]:
df_1_2_3_agg = df_1_2_3.groupby(['studyId_0831', 'recent_date', 'age', 'gender']).agg({
    'consult': lambda x: ' '.join(x.dropna()) if not x.isna().all() else None,  # Unisci i consulti separati da spazio, mantieni NaN se non ci sono consulti
    #'delta': 'first'  # Mantieni il primo valore di delta (puoi cambiarlo se desideri altro comportamento)
}).reset_index()

df_1_2_3_agg

## 4. Aggiunta delle GP Letters

In [ ]:
df_letters = pd.read_parquet(r'L:\lab_research\RES-Folder-UPOD\ODIN-UC4\E_ResearchData\2_ResearchData\20241118\refletters_deid.parquet')

df_letters['studyId_0831'] = df_letters['studyId_0831'].astype('uint64')

df_letters.rename(columns={'created': 'date'}, inplace=True)

df_letters['date'] = df_consults['date'].dt.normalize()

df_letters = df_letters.drop_duplicates()

df_letters.rename(columns={'stripped_text': 'letter'}, inplace=True)

df_letters

In [ ]:
num_missing_in_letters = len(set(df_afsprak_recents['studyId_0831']) - set(df_letters['studyId_0831']))

missing_in_letters = list(set(df_afsprak_recents['studyId_0831']) - set(df_letters['studyId_0831']))

print(f"Patients with missing GP letters ({num_missing_in_letters}): \n{missing_in_letters}")

In [ ]:
df_1_2_3_4 = pd.merge(
    left = df_1_2_3_agg,
    right = df_letters,
    on = 'studyId_0831',
    how = 'left'
)

df_1_2_3_4

In [ ]:
# Calcolo la differenza in giorni tra l'ultima data e tutte le altre date per ogni paziente
df_1_2_3_4['delta'] = (df_1_2_3_4['recent_date'] - df_1_2_3_4['date']).dt.days

df_1_2_3_4

In [ ]:
df_1_2_3_4.loc[~df_1_2_3_4['delta'].between(0, MAX_TIME, inclusive='both'), 'letter'] = None

df_1_2_3_4

In [ ]:
len(set(df_afsprak_recents['studyId_0831']) - set(df_1_2_3_4['studyId_0831']))

In [ ]:
df_1_2_3_4_agg = df_1_2_3_4.groupby(['studyId_0831', 'recent_date', 'age', 'gender', 'consult'], dropna=False).agg({
    'letter': lambda x: ' '.join(x.dropna()) if not x.isna().all() else None,  # Unisci i consulti separati da spazio, mantieni NaN se non ci sono consulti
    #'delta': 'first'  # Mantieni il primo valore di delta (puoi cambiarlo se desideri altro comportamento)
}).reset_index()

df_1_2_3_4_agg

In [ ]:
len(set(df_afsprak_recents['studyId_0831']) - set(df_1_2_3_4_agg['studyId_0831']))

## 5. Aggiunta degli embedding dei farmaci

In [ ]:
df_med_emb = pd.read_parquet(r'T:\lab_research\RES-Folder-UPOD\ODIN-UC4\G_Output\2_Data\parquet\df_medicatie_embeddings.parquet')

# Rimuovo l'orario dalla data
df_med_emb['date'] = df_med_emb['date'].dt.normalize()

df_med_emb = df_med_emb.sort_values(by=['studyId_0831', 'date'])

df_med_emb = df_med_emb.drop(columns='code5_ATC_code')

df_med_emb.columns = [f"medication_{col}" if col.startswith('D') else col for col in df_med_emb.columns]

df_med_emb

In [ ]:
num_missing_in_medication = len(set(df_afsprak_recents['studyId_0831']) - set(df_med_emb['studyId_0831']))

missing_in_medication = list(set(df_afsprak_recents['studyId_0831']) - set(df_med_emb['studyId_0831']))

print(f"Patients with missing medications ({num_missing_in_medication}): \n{missing_in_medication}")

In [ ]:
df_1_2_3_4_5 = pd.merge(
    left = df_1_2_3_4_agg,
    right = df_med_emb,
    on = 'studyId_0831',
    how = 'left'
)

df_1_2_3_4_5

In [ ]:
len(set(df_afsprak_recents['studyId_0831']) - set(df_1_2_3_4_5['studyId_0831']))

In [ ]:
# Calcolo la differenza in giorni tra l'ultima data e tutte le altre date per ogni paziente
df_1_2_3_4_5['delta'] = (df_1_2_3_4_5['recent_date'] - df_1_2_3_4_5['date']).dt.days

df_1_2_3_4_5

In [ ]:
len(set(df_afsprak_recents['studyId_0831']) - set(df_1_2_3_4_5['studyId_0831']))

In [ ]:
df_1_2_3_4_5.loc[~df_1_2_3_4_5['delta'].between(0, MAX_TIME, inclusive='both'), 'letter'] = None

df_1_2_3_4_5

In [ ]:
df_1_2_3_4_5_agg = df_1_2_3_4_5.groupby('studyId_0831', dropna=False)\
    .apply(avg_mean_emb).reset_index(drop=True)

df_1_2_3_4_5_agg

## 6) Aggiunta delle labels

In [ ]:
df_merged = pd.merge(
    left = df_1_2_3_4_5_agg,
    right = labels,
    on = 'studyId_0831',
    how = 'left'
)

df_merged

In [ ]:
df_merged.drop(columns='recent_date', inplace=True)

df_merged.to_csv(r'T:\lab_research\RES-Folder-UPOD\ODIN-UC4\G_Output\2_Data\csv\df_all_features.csv', index=False)

---

# Removing referral letters
Since there are too few samples with referral letters (see `dataset_analyse`), it has been decided to keep only the samples that have both the consultation and the medications.

In [ ]:
df_merged_filt = df_merged.dropna(subset=['consult', 'medication_D0']).reset_index(drop=True)

labels = df_merged_filt['label']

consults = df_merged_filt['consult']

df_merged_filt.drop(columns=['letter'], inplace=True)

df_merged_filt

In [ ]:
df_merged_filt.to_parquet(r'T:\lab_research\RES-Folder-UPOD\ODIN-UC4\G_Output\2_Data\parquet\df_consults_medication.parquet')